<a href="https://colab.research.google.com/github/JorgeEncinas/pytorch_ztm/blob/main/07_pytorch_experiment_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. PyTorch Experiment Tracking

Machine Learning is very experimental. To figure out which experiments are worth it, we got `Experiment Tracking`.

It helps us point out what works or doesn't.

We'll work with programmatically tracking experiments

In [ ]:
#!nvidia-smi

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms

try:
  from torchinfo import summary
except:
  print("[INFO] couldn't find torchinfo... installing....")
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from going_modular.going_modular import data_setup, engine
except:
  print("[INFO] Couldn't find going modular scripts, downloading....")
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine

In [ ]:
def set_seeds(seed: int=42):
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)

In [ ]:
set_seeds()

# 1. Get Data
We'll get the pizza, steak, sushi images

In [ ]:
import os
import zipfile
from pathlib import Path

import requests

def download_images(
    source: str,
    destination: str,
    remove_source: bool = True
) -> Path:
  """Downloads a zipped dataset from Source and unzips to destination
  """
  # Setup path
  data_path = Path("data/")
  image_path = data_path / destination

  if image_path.is_dir() and len(list(image_path.glob("*/*/*.jpg"))) > 5:
    print(f"[INFO] {image_path} directory already exists, skipping download")
  else:
    print(f"[INFO] did not find '{image_path}' directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    target_file = Path(source).name
    with open(data_path / target_file, "wb") as f:
      req = requests.get(source)
      print(f"[INFO] downloading '{target_file}' from '{source}'")
      f.write(req.content)

    with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
      print(f"Unzipping '{target_file}'...")
      zip_ref.extractall(image_path)

    if remove_source:
      os.remove(data_path / target_file)
  return image_path

In [ ]:
image_path = download_images(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"
)

# 2. Creating DataLoaders with manual transforms

The goal with transforms is ensuring your custom data is formatted ina reproducible way that is usable with pretrained models (whatever your choice of pretrained model).

In [ ]:
# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"

In [ ]:
# Setup ImageNet normalization levels
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

from torchvision import transforms
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])
print(f"Manually created transforms: {manual_transforms}")

# Create DataLoaders
from going_modular.going_modular import data_setup
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms,
    batch_size=32
)
train_dataloader, test_dataloader, class_names

## 2.2 Creating DataLoaders via Automatic Transforms

In [ ]:
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT #.IMAGENET1K_V1
auto_transforms = weights.transforms()

train_dir = image_path / "train"
test_dir = image_path / "test"

train_dataloader_auto, test_dataloader_auto, class_names_auto = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transforms,
    batch_size=32
)
train_dataloader_auto, test_dataloader_auto, class_names_auto

# 3. Getting a pretrained model, freezing the base layers, changing the `classifier` head

In [ ]:
model_en = efficientnet_b0(weights=weights).to(device)
model_en

In [ ]:
model_en.classifier

In [ ]:
for param in model_en.features.parameters():
  param.requires_grad = False

In [ ]:
model_en.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names), bias=True)
)
model_en.classifier

In [ ]:
from torchinfo import summary
summary(model_en,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

# 4. Train a single model and track results

In [ ]:
import torch
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()
writer

In [ ]:
from going_modular.going_modular.engine import train_step, test_step
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

In [ ]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for
    each epoch.
    In the form: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]}
    For example if training for epochs=2:
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]}
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # Make sure model on target device
    model.to(device)

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        #New: Experiment Tracker
        writer.add_scalars(
            main_tag = "Loss",
            tag_scalar_dict={
                "train_loss":train_loss,
                "test_loss":test_loss
            },
            global_step=epoch
        )
        writer.add_scalars(
            main_tag="Accuracy",
            tag_scalar_dict={
                "train_acc":train_acc,
                "test_acc":test_acc
            },
            global_step=epoch
        )

        writer.add_graph( #To see the computations our model goes through
          model=model_en,
          input_to_model=torch.randn(32, 3, 224, 224).to(device)
        )

    # Close the writer after the loop
    writer.close()
    # Return the filled results at the end of the epochs
    return results

In [ ]:
set_seeds()
optimizer = torch.optim.Adam(
    params=model_en.parameters(),
    lr=0.001
)
loss_fn = nn.CrossEntropyLoss()

results = train(
    model=model_en,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device
)

# 5. View our model's results using `TensorBoard`

You can view TensorBoard results in various ways, check out the documentation

In [ ]:
!pip show tensorboard

In [ ]:
# From within the notebook
%load_ext tensorboard
%tensorboard --logdir runs #This comes from the `runs` folder that was created by the writer.

# 6. Creating a function to prepare a `SummaryWriter()` instance

By default, our `SummaryWriter()` class saves to `log_dir`

What if we wanted to save different experiments to different folder, so that we have an easy way to tell them apart?

For example, we'd like to track:
  - Experiment date/timestamp
  - Experiment name
  - Model name
  - Extra - is there anything else that should be tracked?

Let's create a function to create a `SummaryWriter()` instance to take all of these things into account

So ideally, we end up tracking experiments to a directory:

`runs/YYY-MM-DD/experiment_name/model_name/extra`

In [ ]:
from datetime import datetime

datetime.now().strftime("%Y-%m-%d")

In [ ]:
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

def create_writer(
    experiment_name : str,
    model_name : str,
    extra: str = None
) -> torch.utils.tensorboard.writer.SummaryWriter:
  """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance tracking to a specific directory
  """

  # Get timestamp of current date in reverse order
  timestamp = datetime.now().strftime("%Y-%m-%d")

  if extra:
    # Create log directory path
    log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
  else:
    log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
  print(f"[INFO] Created SummaryWriter saving to {log_dir}")
  return SummaryWriter(log_dir)

In [ ]:
example_writer = create_writer(
    experiment_name="data_10_percent",
    model_name = "effnetb0",
    extra="5_epochs"
)
example_writer

## 6.1 Update the `train()` method to include a `writer` parameter


In [ ]:
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

from going_modular.going_modular.engine import train_step, test_step

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer : torch.utils.tensorboard.writer.SummaryWriter) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for
    each epoch.
    In the form: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]}
    For example if training for epochs=2:
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]}
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # Make sure model on target device
    model.to(device)

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        #New: Experiment Tracker
        if writer:
          writer.add_scalars(
              main_tag = "Loss",
              tag_scalar_dict={
                  "train_loss":train_loss,
                  "test_loss":test_loss
              },
              global_step=epoch
          )
          writer.add_scalars(
              main_tag="Accuracy",
              tag_scalar_dict={
                  "train_acc":train_acc,
                  "test_acc":test_acc
              },
              global_step=epoch
          )

          writer.add_graph( #To see the computations our model goes through
            model=model_en,
            input_to_model=torch.randn(32, 3, 224, 224).to(device)
          )
    if writer:
      # Close the writer after the loop
      writer.close()
    # Return the filled results at the end of the epochs
    return results

# 7. Setting up a series of modeling experiments

* Setup 2x modeling experiments with `effnetb0`, and train one model for 5 epochs, another for 10 epochs.

You could create a list of epoch values and then run training for each one.

## 7.1 What kind of experiments should you run?

The number of ML Experiments you should run is like the number of different models you can build... almost limitless.

However, you can't test everything, so what should we test?
  - Change the number of epochs
  - Change the number of hidden layers/units
  - Change the amount of data (right now we're using 10% of the Food101 dataset)
  - Change the Learning Rate
  - Try different kinds of Data Augmentation
  - Choose a different model architecture


There is no silver bullet here.

You should try all of these plus any Hyperparameters you can think of. That's why Transfer Learning is a great tool you should use: It is a working model you can apply to your own problem.

Keep in mind we'll start designing experiments over these.

## 7.2 What experiments are we going to run?

We'll keep it simple, but think of every setting as a dial in ML. We will play with only 3 of them

1. Model size - EffNet B0 vs B2
2. Dataset size - 105 of pizza, steak, sushi images vs 20%
3. Training time - 5 epochs vs 10 epochs.

In the instructor's experience, these factors are quite effective or influential.

Generally, more data, better results; and longer training time is better to a point (I guess it overfits after that)

## 7.3 Download different datasets

We want 2 datasets

1. Pizza, steak, sushi 10%
2. Pizza, steak, sushi 20%



In [ ]:
data_10_pct_path = download_images(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"
)

data_20_pct_path = download_images(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi_20_percent.zip",
    destination="pizza_steak_sushi_20_pct"
)

In [ ]:
from torchvision import datasets

In [ ]:
train_dir_10_pct = data_10_pct_path / "train"
test_dir_10_pct = data_10_pct_path / "test"

train_dir_20_pct = data_20_pct_path / "train"
test_dir_20_pct = data_20_pct_path / "test"

In [ ]:
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights
auto_transforms = EfficientNet_B0_Weights.DEFAULT

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

manual_transforms_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

In [ ]:
#train_10_pct_data = datasets.ImageFolder(
#    root=train_dir_10_pct,
#    transform=manual_transforms_2
#)
#test_10_pct_data = datasets.ImageFolder(
#    root=test_dir_10_pct,
#    transform=manual_transforms_2
#)

# Or...
from going_modular.going_modular import data_setup

BATCH_SIZE=32

train_dataloader_10pct, test_dataloader_10pct, class_names = data_setup.create_dataloaders(
    train_dir = train_dir_10_pct,
    test_dir=test_dir_10_pct,
    transform=manual_transforms_2,
    batch_size = BATCH_SIZE
)

train_dataloader_20pct, test_dataloader_20pct, class_names = data_setup.create_dataloaders(
    train_dir = train_dir_20_pct,
    test_dir=test_dir_20_pct,
    transform=manual_transforms_2,
    batch_size = BATCH_SIZE
)

In [ ]:
print(f"10% - {BATCH_SIZE} samples. Tr: {len(train_dataloader_10pct)} b. | Te: {len(test_dataloader_10pct)} b.")
print(f"20% - {BATCH_SIZE} samples. Tr: {len(train_dataloader_20pct)} b. | Te: {len(test_dataloader_20pct)} b.")

In [ ]:
from torchvision.models import efficientnet_b0, efficientnet_b2, EfficientNet_B0_Weights, EfficientNet_B2_Weights

In [ ]:
def create_effnetb0(output_classes : int, seed: int = None):
  if seed:
    set_seeds()
  else:
    set_seeds()
  weights = EfficientNet_B0_Weights.DEFAULT
  model = efficientnet_b0(weights=weights)
  model.classifier = nn.Sequential(
      nn.Dropout(p=0.2, inplace=True),
      nn.Linear(in_features=1280, out_features=output_classes, bias=True)
  )
  for param in model.features.parameters():
    param.requires_grad = False
  model_transforms = weights.transforms()
  return model, model_transforms

def create_effnetb2(output_classes : int, seed: int = None):
  if seed:
    set_seeds()
  else:
    set_seeds()
  weights = EfficientNet_B2_Weights.DEFAULT
  model = efficientnet_b2(weights=weights)
  model.classifier = nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=output_classes, bias=True)
  )
  for param in model.features.parameters():
    param.requires_grad = False
  model_transforms = weights.transforms()
  return model, model_transforms

In [ ]:
weights_example = EfficientNet_B0_Weights.DEFAULT
model_example = efficientnet_b0(weights=weights)
model_example.classifier

In [ ]:
  weights_example = EfficientNet_B2_Weights.DEFAULT
  model_example = efficientnet_b2(weights=weights_example)
  model_example.classifier

In [ ]:
OUT_FEATURES = len(class_names)
model_b0_1, mb0_tf = create_effnetb0(OUT_FEATURES)
model_b2_1, mb2_tf = create_effnetb2(OUT_FEATURES)

In [ ]:
summary(model_b2_1)

## 7.6 Create Experiments and setup training code

In [ ]:
# Create epoch list
num_epochs = [5, 10]

# Create models list (need to create a new model for each experiment)
models = ["effnetb0", "effnetb2"]

# Create a DataLoaders dictionary
train_dataloaders = {
    "data_10_pct": train_dataloader_10pct,
    "data_20_pct": train_dataloader_20pct
}
test_dataloaders = {
    "data_10_pct": test_dataloader_10pct,
    "data_20_pct": test_dataloader_20pct
}

In [ ]:
from going_modular.going_modular.utils import save_model

In [ ]:
%%time
# These experiments shouldn't take too long to run, so let's check how they do.
from going_modular.going_modular.utils import save_model

# set seeds
set_seeds(seed=42)

# Keep track of experiment numbers
experiment_number = 0

# Loop through each dataloader
for dataloader_name, train_dataloader in train_dataloaders.items():
  # Loop through the epochs
  for epochs in num_epochs:
    # Loop through each model name and create a new model instance
    for model_name in models:
      experiment_number +=1
      print(f"[INFO] {experiment_number} | M: {model_name} | DL: {dataloader_name} | Ep: {epochs}")

      if model_name == "effnetb0":
        model, tf = create_effnetb0(output_classes=len(class_names), seed=42)
      else:
        model, tf = create_effnetb2(output_classes=len(class_names), seed=42)

      # Loss and Optimizer
      optimizer = torch.optim.Adam(
          params = model.parameters(),
          lr=0.001
      )
      loss_fn = nn.CrossEntropyLoss()

      m_results = train(
          model=model,
          train_dataloader=train_dataloader,
          test_dataloader=test_dataloader,
          optimizer=optimizer,
          loss_fn=loss_fn,
          epochs=epochs,
          device=device,
          writer=create_writer(
              experiment_name=dataloader_name,
              model_name=model_name,
              extra=f"{epochs}_epochs"
          )
      )

      # Save the model to file so we can import it later if need be
      save_filepath = f"07_{model_name}_{dataloader_name}_{epochs}_epochs.pth"
      save_model(
          model=model,
          target_dir="models",
          model_name=save_filepath
      )
      print("-"*50 + "\n")

# 8. View Experiments in TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs #Log Directory

# 9. Load in the best model and make predictions with it

In [ ]:
import torch
from torch import nn

In [ ]:
best_model_path = "models/07_effnetb0_data_20_pct_10_epochs.pth"

best_model, bm_tf = create_effnetb0(output_classes=len(class_names))

best_model.load_state_dict(torch.load(best_model_path))

Our goal is to create a `FoodVision Mini` model that performs well enough and is able to run in the device / web browser

In [ ]:
# Check the model file size. A `Sound check`
from pathlib import Path

# Get the model size in bytes then convert it to megabytes
effnetb2_model_size = Path(best_model_path).stat().st_size // (1024) # is in Bytes, so division transforms to MB
print(f"EfficientNetB2 feature extractor model size: {effnetb2_model_size} MB")

In [ ]:
from going_modular.going_modular.predictions import pred_and_plot_image

In [ ]:
import random
num_images_to_plot = 1
test_image_path_list = list(Path(data_20_pct_path).glob("*/*/*.jpg"))

test_img_path_samples = random.sample(test_image_path_list, k=num_images_to_plot)
print(class_names)
for image_path in test_img_path_samples:
  pred_and_plot_image(
      model=best_model,
      image_path=image_path,
      class_names=class_names,
      image_size=(224, 224)
  )

In [ ]:
import requests

def download_image(img_filename : str, url: str):
  # Setup custom image path
  custom_image_path = Path("data/") / img_filename

  # Download the image if it doesn't exist
  if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
      # When downloading from Github, get the Raw link.
      request = requests.get(url)
      print(f"Downloading {custom_image_path}... ")
      f.write(request.content)
  else:
    print(f"{custom_image_path} already exists, skipping download")
  return custom_image_path

In [ ]:
pizza_dad_img = download_image("04_pizza_dad.jpeg", "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/images/04-pizza-dad.jpeg")


pred_and_plot_image(
    model=best_model,
    image_path=pizza_dad_img,
    class_names = class_names,
    image_size=(224, 224)
)

# Metadata Cleaning

In [ ]:
import nbformat

notebook_path = '/content/drive/MyDrive/Colab Notebooks/07_pytorch_experiment_tracking.ipynb'

with open(notebook_path, "r", encoding="utf-8") as f:
  nb = nbformat.read(f, as_version=4)

if "widgets" in nb["metadata"]:
  del nb["metadata"]["widgets"]
  print("Removed widget metadata")
else:
  print("No widget metadata found")

with open(notebook_path, "w", encoding="utf-8") as f:
  nbformat.write(nb, f)
  print("Notebook cleaned and saved back to drive.")